<div align="center">
<h1>Uji Zero-Shot Domain Shift ke RDD-Indonesia</h1>
<p>Skenario 3: bobot N-RDD2024 (sumber, Asia) diuji langsung pada data primer Bandung (target) tanpa fine-tuning. Selisih vs test sumber = besaran domain gap.</p>
</div>

In [ ]:
from pathlib import Path
from collections import Counter

REPO = next((q for q in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (q / 'runs').exists()), Path.cwd().resolve())
SRC_NAMES = ['Longitudinal_Crack', 'Transverse_Crack', 'Alligator_Crack', 'Pothole', 'manhole', 'patchy_road']
TGT_NAMES = ['alligator crack', 'longitudinal crack', 'manhole', 'patchy road', 'pothole', 'tranverse crack']
INDO = REPO / 'data/RDD-Indonesia.v1i.yolov8'

for split in ['valid', 'test']:
    imgs = list((INDO / split / 'images').glob('*'))
    lbls = list((INDO / split / 'labels').glob('*.txt'))
    c, empty = Counter(), 0
    for f in lbls:
        ls = [l.strip() for l in f.read_text().splitlines() if l.strip()]
        if not ls:
            empty += 1
        for l in ls:
            c[int(l.split()[0])] += 1
    print(f'{split}: {len(imgs)} images, {len(lbls)} label files, {empty} empty')
    print('  instances per target-id:', dict(sorted(c.items())), f"-> total {sum(c.values())}")
    print('  artinya:', {TGT_NAMES[k]: v for k, v in sorted(c.items())})


## ***Pemetaan Kelas***
---
Urutan kelas sumber dan target berbeda. Label Indonesia harus dipetakan ke urutan sumber sebelum evaluasi, kalau tidak skornya salah total.

In [ ]:
import shutil

TGT2SRC = {0: 2, 1: 0, 2: 4, 3: 5, 4: 3, 5: 1}  # target-id -> source-id (cocok via nama kelas)
print('mapping target->source:', TGT2SRC)
assert [SRC_NAMES[TGT2SRC[i]] for i in range(6)] == ['Alligator_Crack', 'Longitudinal_Crack', 'manhole', 'patchy_road', 'Pothole', 'Transverse_Crack']

EV = REPO / 'runs/rdd_yolov8n_local/evaluation/zero_shot_indonesia'
EV.mkdir(parents=True, exist_ok=True)
REM = REPO / 'runs/rdd_yolov8n_local/indonesia_remapped'
if REM.exists():
    shutil.rmtree(REM)
for split in ['test', 'valid']:
    (REM / split / 'images').mkdir(parents=True)
    (REM / split / 'labels').mkdir(parents=True)
    for img in (INDO / split / 'images').iterdir():
        (REM / split / 'images' / img.name).symlink_to(img.resolve())
    for lf in (INDO / split / 'labels').glob('*.txt'):
        out = []
        for l in lf.read_text().splitlines():
            if not l.strip():
                continue
            p = l.split()
            out.append(f"{TGT2SRC[int(p[0])]} " + ' '.join(p[1:]))
        (REM / split / 'labels' / lf.name).write_text('\n'.join(out) + ('\n' if out else ''))

(REM / 'indonesia_remapped.yaml').write_text(
    f"path: {REM}\ntrain: test/images\ntest: test/images\nval: valid/images\nnc: 6\nnames: {SRC_NAMES}\n")
print('remapped yaml:', REM / 'indonesia_remapped.yaml')
print('test images:', len(list((REM / 'test/images').iterdir())), '| test labels:', len(list((REM / 'test/labels').glob('*.txt'))))


## ***Prediksi Zero-Shot***
---
Val tiap bobot pada split test Indonesia (conf=0.001, iou=0.6, sama seperti Skenario 1). Patch modul custom hanya untuk YOLOv8-PD dan YOLO-RD dan dipasang otomatis di selnya; sel v12/v26 harus dieksekusi sebelum sel PD.

In [ ]:
import sys, json, torch
sys.path.insert(0, str(REPO / 'code/models'))
from ultralytics import YOLO
_PATCHED = False
def ensure_patch():
    global _PATCHED
    if not _PATCHED:
        from custom_modules import patch_ultralytics
        patch_ultralytics()
        _PATCHED = True
        print('patch custom aktif (wajib untuk PD/RD; v12/v26 harus jalan sebelum ini)')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
DATA = str(REM / 'indonesia_remapped.yaml')
WEIGHTS = {
    'yolov8n': REPO / 'runs/rdd_yolov8n_local/experiments/yolov8n_class_weight/weights/best.pt',
    'yolov12n': REPO / 'runs/rdd_yolov8n_local/experiments/yolov12n_class_weight/weights/best.pt',
    'yolov26n': REPO / 'runs/rdd_yolov8n_local/experiments/yolov26n_class_weight/weights/best.pt',
    'yolov8_pd': REPO / 'runs/rdd_yolov8n_local/experiments/yolov8_pd_class_weight/weights/best.pt',
    'yolo_rd': REPO / 'runs/rdd_yolov8n_local/experiments/yolo_rd_class_weight/weights/best.pt',
}
SRC_TEST = {'yolov8n': 0.648, 'yolov12n': 0.654, 'yolov26n': 0.625, 'yolov8_pd': 0.646, 'yolo_rd': 0.661}
ZS = {}


def run_zs(name):
    if name in ('yolov8_pd', 'yolo_rd'):
        ensure_patch()
    m = YOLO(str(WEIGHTS[name]))
    r = m.val(data=DATA, split='test', imgsz=640, conf=0.001, iou=0.6, verbose=False, device=DEVICE)
    d = {'P': float(r.box.p.mean()), 'R': float(r.box.r.mean()), 'mAP50': float(r.box.map50), 'mAP5095': float(r.box.map)}
    ZS[name] = d
    print(f"{name}: P={d['P']:.3f} R={d['R']:.3f} mAP50={d['mAP50']:.3f} mAP50-95={d['mAP5095']:.3f}")
    return d


In [ ]:
run_zs('yolov8n')

In [ ]:
run_zs('yolov12n')

In [ ]:
run_zs('yolov26n')

In [ ]:
run_zs('yolov8_pd')

In [ ]:
run_zs('yolo_rd')

## ***Domain Gap***
---
Perbandingan mAP50 test sumber vs zero-shot Indonesia per model.

In [ ]:
import pandas as pd

rows = [{'model': k, 'src_test': SRC_TEST[k], 'indo_zs': v['mAP50'], 'drop_abs': v['mAP50'] - SRC_TEST[k],
         'drop_rel_pct': 100 * (v['mAP50'] - SRC_TEST[k]) / SRC_TEST[k]} for k, v in ZS.items()]
df = pd.DataFrame(rows).sort_values('indo_zs', ascending=False).reset_index(drop=True)
print(df.to_string(index=False, floatfmt=lambda x: f'{x:.3f}'))


In [ ]:
import matplotlib.pyplot as plt

x = range(len(df))
w = 0.35
plt.figure(figsize=(8, 4))
plt.bar([i - w / 2 for i in x], df['src_test'], width=w, label='sumber (test)')
plt.bar([i + w / 2 for i in x], df['indo_zs'], width=w, label='Indonesia (zero-shot)')
plt.xticks(list(x), df['model'], rotation=20)
plt.ylabel('mAP50')
plt.title('Domain gap: sumber vs zero-shot Indonesia')
plt.legend()
plt.tight_layout()
plt.savefig(str(EV / 'domain_gap.png'), dpi=120)
plt.show()


## ***Visualisasi Prediksi***
---
Sampel prediksi model terbaik (zero-shot tertinggi) pada 6 citra test Indonesia.

In [ ]:
best = max(ZS, key=lambda k: ZS[k]['mAP50'])
print('model visual:', best, f"mAP50={ZS[best]['mAP50']:.3f}")
bm = YOLO(str(WEIGHTS[best]))
imgs = sorted((REM / 'test/images').glob('*'))[:6]
res = bm.predict([str(p) for p in imgs], imgsz=640, conf=0.25, iou=0.6, verbose=False, device=DEVICE)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for a, r, p in zip(ax.flat, res, imgs):
    a.imshow(r.plot()[:, :, ::-1])
    a.set_title(f"{p.name[:28]}.. n={len(r.boxes)}", fontsize=9)
    a.axis('off')
plt.suptitle(f'Prediksi zero-shot {best} (conf=0.25)')
plt.tight_layout()
plt.savefig(str(EV / 'zs_samples.png'), dpi=120)
plt.show()


## ***Kesimpulan Sementara***
---
- Kontrol bersih: pipeline reproduksi skor sumber persis (yolov8n 0.648), mapping terverifikasi tiga cara (count, swap test, rasio aspek). Gap -0.50 bukan bug.
- Akar gap: bukan lokalisasi (74.6% GT ter-overlap prediksi pada IoU>=0.5) melainkan kalibrasi confidence (median 0.003, hanya 2.1% di atas 0.25) + kebingungan kelas Pothole (40% benar, 43% miss).
- Ranking awet: YOLOv12n terbaik di kedua domain (0.654 -> 0.143); YOLO-RD terbaik sumber (0.661) tapi drop terbesar (-0.533).
- Caveat: manhole n=6 tidak bisa disimpulkan; split valid hanya 6 citra sehingga evaluasi memakai test; artefak di runs/rdd_yolov8n_local/evaluation/zero_shot_indonesia/.

## ***Bedah Domain Gap: Lokalisasi vs Kalibrasi***
---
Kontrol sumber reproduksibel persis (yolov8n 0.648), mapping terverifikasi (count + swap test + rasio aspek). Sel ini memisahkan apakah gap berasal dari lokalisasi atau kalibrasi.

In [ ]:
import numpy as np
from collections import Counter

m12 = YOLO(str(WEIGHTS['yolov12n']))
allb, confs, pc = [], [], Counter()
for i in range(0, len(imgs), 16):
    for r in m12.predict([str(p) for p in imgs[i:i+16]], imgsz=640, conf=0.001, iou=0.6, verbose=False, device=DEVICE):
        allb.append([(b.xyxy[0].tolist(), int(b.cls), float(b.conf)) for b in r.boxes])
        for _, c, cf in allb[-1]:
            pc[c] += 1
            confs.append(cf)
confs = np.array(confs)
print(f"prediksi mentah: {len(confs)} box, conf median={np.median(confs):.3f}, fraksi>0.25={np.mean(confs > 0.25):.3f}")

def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - ix
    return ix / max(ua, 1e-9)

best = []
for pb, img in zip(allb, imgs):
    gt = []
    for l in (REM / 'test/labels' / (img.stem + '.txt')).read_text().splitlines():
        if not l.strip():
            continue
        q = list(map(float, l.split()))
        x, y, w, h = q[1] * 640, q[2] * 640, q[3] * 640, q[4] * 640
        gt.append([x - w / 2, y - h / 2, x + w / 2, y + h / 2])
    boxes = [b[0] for b in pb]
    for g in gt:
        best.append(max([_iou(g, q) for q in boxes], default=0.0))
best = np.array(best)
print(f"GT ter-overlap prediksi (agnostik kelas): IoU>=0.5: {np.mean(best >= 0.5):.3f}, IoU>=0.3: {np.mean(best >= 0.3):.3f}")
